# HDF5 → 6-hour NetCDF, cropped to reference grid

Configure paths and dataset mapping, inspect one file, then run a single-day test before enabling all days. Assumes source and reference `x/y` are in the **same coordinate system** and the source grid is rectilinear. Cropping selects source pixels within the reference extent; it does not resample.

In [6]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import xarray as xr
import h5py

BASE_DIR = Path('/Volumes/Rain/new_radar_hdf5_sample')
REFERENCE_NC = Path('../../data/example_grids/eg_radar_grid.nc')
OUTPUT_DIR = Path('/Users/matthewwright/Downloads/test_hdf_files')
HDF5_GLOB = '*.h5'  # change to '*.hdf5' if needed
DATASET_PATH = '/data'  # path to 2D or 3D array in HDF5, e.g. '/Grid/precipitation'
VARIABLE_NAME = 'measurement'
SOURCE_X_PATH = '/x'  # 1D x coordinate array in HDF5
SOURCE_Y_PATH = '/y'  # 1D y coordinate array in HDF5
REF_X = 'x'
REF_Y = 'y'
# Regex must capture YYYYMMDD and HHMM, e.g. ...20250115_0030...
TIMESTAMP_REGEX = r'(?P<date>\d{8})[_T-]?(?P<hhmm>\d{4})(?!\d)'
FOLDER_REGEX = r"\d{4}-\d{2}-\d{2}"
TEST_MODE = True
TEST_DATE = None  # e.g. '20250115'; None = first available day
OVERWRITE = False
STRICT_72 = False  # True: skip windows without exactly 72 unique files


## Inspect reference and sample HDF5
Run this first and adjust the configuration above. Nested HDF5 paths are printed.

In [7]:
def _valid_date(s):
    try:
        pd.to_datetime(s, format='%Y%m%d', errors='raise')
        return True
    except ValueError:
        return False

day_dirs = sorted(
    p for p in BASE_DIR.iterdir()
    if p.is_dir() and re.fullmatch(FOLDER_REGEX, p.name)
)
print(f'Found {len(day_dirs)} daily folders:', [p.name for p in day_dirs[:8]])
with xr.open_dataset(REFERENCE_NC) as ref:
    print('Reference:', ref)
    print('Reference x/y ranges:', (float(ref[REF_X].min()), float(ref[REF_X].max())), (float(ref[REF_Y].min()), float(ref[REF_Y].max())))
if not day_dirs:
    raise FileNotFoundError('No YYYYMMDD daily folders found')
sample_files = sorted(day_dirs[0].rglob(HDF5_GLOB))
if not sample_files:
    raise FileNotFoundError('No matching HDF5 files in first daily folder')
print('Sample:', sample_files[0])
with h5py.File(sample_files[0], 'r') as f:
    def describe(name, obj):
        if isinstance(obj, h5py.Dataset): print(name, obj.shape, obj.dtype)
    f.visititems(describe)


Found 4 daily folders: ['2025-04-30', '2025-05-01', '2025-05-02', '2025-05-03']
Reference: <xarray.Dataset> Size: 4MB
Dimensions:      (y: 396, x: 396)
Coordinates:
  * y            (y) float64 3kB -1.98e+05 -1.97e+05 ... 1.96e+05 1.97e+05
  * x            (x) float64 3kB -1.98e+05 -1.97e+05 ... 1.96e+05 1.97e+05
    lon          (y, x) float64 1MB ...
    lat          (y, x) float64 1MB ...
    central_lon  float64 8B ...
    central_lat  float64 8B ...
    time         datetime64[ns] 8B ...
Data variables:
    qpe          (y, x) float64 1MB ...
Reference x/y ranges: (-198000.0, 197000.0) (-198000.0, 197000.0)
Sample: /Volumes/Rain/new_radar_hdf5_sample/2025-04-30/vol_2025-04-30_10-17-55.h5
dataset1/data1/data (360, 569) uint8
dataset1/data10/data (360, 569) uint8
dataset1/data11/data (360, 569) uint8
dataset1/data12/data (360, 569) uint8
dataset1/data13/data (360, 569) uint8
dataset1/data2/data (360, 569) uint8
dataset1/data3/data (360, 569) uint8
dataset1/data4/data (360, 569) uint

In [8]:
import h5py
import numpy as np

sample_file = day_dirs[0] / "vol_2025-04-30_10-17-55.h5"

with h5py.File(sample_file, "r") as f:

    for group_path in [
        "what",
        "where",
        "how",
        "dataset1/what",
        "dataset1/where",
        "dataset1/how",
    ]:
        if group_path in f:
            print(f"\n--- {group_path} ---")
            for key, value in f[group_path].attrs.items():
                print(f"{key}: {value}")

    print("\n--- Radar quantities ---")

    for i in range(1, 14):
        path = f"dataset1/data{i}/what"

        if path in f:
            print(f"\n{path}")
            for key, value in f[path].attrs.items():
                print(f"  {key}: {value}")


--- what ---
date: b'20250430'
object: b'PVOL'
source: b'RAD:rwanda'
time: b'101755'
version: b'H5rad 2.3'

--- where ---
height: 0.0
lat: -2.158
lon: 30.1131097

--- how ---
beamwH: 1
beamwV: 1
beamwidth: 1
wavelength: 5.329643697777778

--- dataset1/what ---
enddate: b'20250430'
endtime: b'101811'
product: b'SCAN'
startdate: b'20250430'
starttime: b'101755'

--- dataset1/where ---
a1gate: 154
elangle: 1.5
nbins: 569
nrays: 360
rscale: 250.0
rstart: 0.25

--- dataset1/how ---
NI: 88.6164035259963
antspeed: 22.0
astart: -0.5
highprf: 950.0
pulsewidth: 97.0
rpm: 3.6666666666666665
scan_count: 15
scan_index: 1
scanplan_tag: b'1746008251'
startazA: [ -0.5   0.5   1.5   2.5   3.5   4.5   5.5   6.5   7.5   8.5   9.5  10.5
  11.5  12.5  13.5  14.5  15.5  16.5  17.5  18.5  19.5  20.5  21.5  22.5
  23.5  24.5  25.5  26.5  27.5  28.5  29.5  30.5  31.5  32.5  33.5  34.5
  35.5  36.5  37.5  38.5  39.5  40.5  41.5  42.5  43.5  44.5  45.5  46.5
  47.5  48.5  49.5  50.5  51.5  52.5  53.5  54.5  55.

## Processing functions
Coordinates are read from HDF5. A 2D field is expected, with shape `(len(y), len(x))`. For multiple variables, adapt `read_cropped` to create more data variables.

In [ ]:
def parse_time(path):
    match = re.search(TIMESTAMP_REGEX, path.name)
    if match is None:
        raise ValueError(f'No timestamp in filename: {path.name}')
    return pd.to_datetime(match.group('date') + match.group('hhmm'), format='%Y%m%d%H%M')

with xr.open_dataset(REFERENCE_NC) as ref:
    ref_x = np.asarray(ref[REF_X].values)
    ref_y = np.asarray(ref[REF_Y].values)
if ref_x.ndim != 1 or ref_y.ndim != 1:
    raise ValueError('This notebook requires 1D reference x/y coordinates')


def read_cropped(path, timestamp):
    with h5py.File(path, 'r') as f:
        x = np.asarray(f[SOURCE_X_PATH][...]).squeeze()
        y = np.asarray(f[SOURCE_Y_PATH][...]).squeeze()
        data = np.asarray(f[DATASET_PATH][...])
    if x.ndim != 1 or y.ndim != 1 or data.shape != (len(y), len(x)):
        raise ValueError(f'Unexpected source dimensions in {path}: data={data.shape}, y={y.shape}, x={x.shape}')
    ix = np.flatnonzero((x >= ref_x.min()) & (x <= ref_x.max()))
    iy = np.flatnonzero((y >= ref_y.min()) & (y <= ref_y.max()))
    if not len(ix) or not len(iy):
        raise ValueError(f'No spatial overlap: {path}')
    # This is a spatial subset of source pixels, NOT interpolation to reference pixels.
    cropped = data[np.ix_(iy, ix)]
    return xr.DataArray(cropped, dims=('y', 'x'), coords={'y': y[iy], 'x': x[ix]}, name=VARIABLE_NAME).expand_dims(time=[timestamp])


def files_for_day(day_dir):
    records = []
    for path in sorted(day_dir.rglob(HDF5_GLOB)):
        try:
            ts = parse_time(path)
            if ts.strftime('%Y%m%d') != day_dir.name:
                print(f'WARNING: timestamp outside folder date: {path.name}')
            records.append((ts, path))
        except ValueError as e:
            print('SKIP:', e)
    return records


def process_window(window_start, records):
    records = sorted(records, key=lambda r: r[0])
    times = [ts for ts, _ in records]
    if len(times) != len(set(times)):
        raise ValueError(f'Duplicate timestamps in {window_start}')
    expected = pd.date_range(window_start, periods=72, freq='5min')
    missing = expected.difference(pd.DatetimeIndex(times))
    unexpected = pd.DatetimeIndex(times).difference(expected)
    print(f'{window_start}: {len(records)}/72 files, missing={len(missing)}, off-grid={len(unexpected)}')
    if len(missing): print('  First missing:', list(missing[:8]))
    if len(unexpected): print('  Off-grid:', list(unexpected[:8]))
    if STRICT_72 and (len(missing) or len(unexpected)):
        print('  Skipped (STRICT_72=True)'); return None
    if not records: return None
    out = OUTPUT_DIR / f'{VARIABLE_NAME}_{window_start:%Y%m%d_%H%M}_6h.nc'
    if out.exists() and not OVERWRITE:
        print('  Exists; skipped:', out); return out
    arrays = [read_cropped(path, ts) for ts, path in tqdm(records, desc=str(window_start), leave=False)]
    # exact alignment catches any HDF5 grid mismatch rather than silently padding.
    ds = xr.concat(arrays, dim='time', join='exact').to_dataset()
    ds.attrs.update(window_start=str(window_start), window_end_exclusive=str(window_start + pd.Timedelta(hours=6)),
                    expected_observations=72, actual_observations=len(records), missing_observations=len(missing),
                    spatial_method='crop source grid to reference x/y extent; no resampling')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    ds.to_netcdf(out, engine='h5netcdf', encoding={VARIABLE_NAME: {'zlib': True, 'complevel': 4}})
    ds.close()
    print('  Wrote:', out)
    return out


## Run: first day or all days
Windows are grouped by the timestamp, not by filename order. This version assumes all 5-minute files for a window reside in the same date folder.

In [ ]:
selected_days = day_dirs[:]
if TEST_MODE:
    selected_days = [next((p for p in day_dirs if p.name == TEST_DATE), day_dirs[0])] if TEST_DATE else day_dirs[:1]
outputs = []
for day_dir in selected_days:
    records = files_for_day(day_dir)
    windows = {}
    for ts, path in records:
        start = ts.floor('6h')
        windows.setdefault(start, []).append((ts, path))
    print(f'\n{day_dir.name}: {len(records)} source files, {len(windows)} populated windows')
    for start, group in sorted(windows.items()):
        result = process_window(start, group)
        if result is not None: outputs.append(result)
print(f'\nGenerated / found {len(outputs)} NetCDF files')


## Validate one output
Inspect shape, timestamps, coordinate ranges, and a quick image of the first time step.

In [ ]:
if outputs:
    with xr.open_dataset(outputs[0]) as check:
        print(check)
        print('Time start/end:', check.time.values[0], check.time.values[-1])
        print('x range:', float(check.x.min()), float(check.x.max()))
        print('y range:', float(check.y.min()), float(check.y.max()))
        check[VARIABLE_NAME].isel(time=0).plot(figsize=(9, 5))
else:
    print('No outputs to validate')


## Important adaptation notes
- **Matching reference pixel centers exactly** requires regridding/interpolation (`interp`, `reindex`, or geospatial reprojection), not just cropping. The notebook currently crops to the reference **bounding box** and keeps original source `x/y` coordinates.
- If the HDF5 uses latitude/longitude 2D arrays, projection coordinates, scaling/offsets, fill values, or transposed arrays, adjust the reader after inspecting the sample.
- If timestamps are in HDF5 attributes rather than filenames, replace `parse_time`.
- Set `TEST_MODE = False` only after validating a sample output.
- Install dependencies if needed: `pip install xarray h5py h5netcdf netCDF4 pandas numpy matplotlib tqdm`.